# Vyolab ComfyUI GPU Worker (Google Colab + Hugging Face Models)

Run this notebook on a Google Colab GPU runtime (T4 / L4 / A100) to serve open-source Hugging Face image & image-to-video models (`FLUX.1-schnell`, `Wan 2.2 TI2V 5B`) to your self-hosted **Vyolab Studio** instance at **$0 API cost**.

When the final cell prints your tunnel URL, set `COMFYUI_BASE_URL=<tunnel_url>` in Vyolab's server-side `.env` and restart Vyolab.

In [ ]:
# 1. Verify GPU and install ComfyUI + huggingface_hub
!nvidia-smi
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt huggingface_hub

In [ ]:
# 2. Download open-source Hugging Face weights for Vyolab workflows
import os
from huggingface_hub import hf_hub_download

# Optional: set HF_TOKEN in Colab Secrets if pulling gated repos
hf_token = os.environ.get('HF_TOKEN') or None

downloads = [
    ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors', '/content/ComfyUI/models/diffusion_models'),
    ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', '/content/ComfyUI/models/text_encoders'),
    ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/vae/wan2.2_vae.safetensors', '/content/ComfyUI/models/vae'),
    ('Comfy-Org/flux1-schnell', 'flux1-schnell-fp8.safetensors', '/content/ComfyUI/models/checkpoints'),
]

for repo, filename, target_dir in downloads:
    os.makedirs(target_dir, exist_ok=True)
    print(f'Downloading {repo}/{filename}...')
    path = hf_hub_download(repo_id=repo, filename=filename, local_dir=target_dir, token=hf_token)
    print('Saved:', path)

In [ ]:
# 3. Launch ComfyUI + Cloudflare quick tunnel for Vyolab COMFYUI_BASE_URL
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, time, re
comfy = subprocess.Popen(['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188'])
time.sleep(8)
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n====================================================')
        print('Set this in Vyolab .env:')
        print(f'COMFYUI_BASE_URL={m.group(0)}')
        print('====================================================\n')
        break
comfy.wait()